<a href="https://colab.research.google.com/github/Digital-AI-Finance/Cryptocurrency/blob/main/03_ethereum_smart_contracts/notebooks/NB02_Gas_Mechanics_Interactive.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Gas Mechanics: Interactive Calculator

**Course:** Build Your Own Cryptocurrency  
**Lesson:** 3 - Ethereum & Smart Contracts

## Learning Objectives
By the end of this notebook, you will:
- Understand the Ethereum gas fee model (Gas Fee = Gas Used x Gas Price)
- Calculate transaction costs for different operation types
- Compare costs across different gas price environments
- Understand the EIP-1559 base fee + priority tip model

In [ ]:
# Setup
!pip install -q matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch
print("Setup complete!")

## The Gas Fee Formula

Every Ethereum transaction requires **gas** - a unit of computational work. The cost model is:

```
Gas Fee (ETH) = Gas Used × Gas Price (Gwei) × 10⁻⁹
```

- **Gas Used**: Determined by the complexity of the operation (fixed per operation type)
- **Gas Price**: Set by the user/market (higher price = faster inclusion in a block)
- **1 Gwei = 10⁻⁹ ETH** (one billionth of an ETH)

In [ ]:
# ========================================
# CHANGE THESE TO EXPERIMENT!
# ========================================
gas_used = 21000          # Simple transfer: 21000, Token transfer: ~65000, Uniswap swap: ~150000
gas_price_gwei = 50       # Low: 10, Medium: 50, High: 200, Surge: 500+
eth_price_usd = 2000      # Current ETH price in USD
# ========================================

# Calculate costs
gas_fee_gwei = gas_used * gas_price_gwei
gas_fee_eth = gas_fee_gwei * 1e-9
gas_fee_usd = gas_fee_eth * eth_price_usd

print("=" * 60)
print("GAS FEE CALCULATOR")
print("=" * 60)
print(f"\n  Gas Used:       {gas_used:>12,} gas units")
print(f"  Gas Price:      {gas_price_gwei:>12} Gwei")
print(f"  ETH Price:      ${eth_price_usd:>11,}")
print(f"\n  {'─' * 40}")
print(f"  Gas Fee:        {gas_fee_gwei:>12,} Gwei")
print(f"  Gas Fee:        {gas_fee_eth:>12.6f} ETH")
print(f"  Gas Fee:        ${gas_fee_usd:>11.2f} USD")
print("=" * 60)

## Gas Costs by Operation Type

Different operations on Ethereum require different amounts of gas. Here's a reference:

| Operation | Gas Used | Description |
|-----------|----------|-------------|
| Simple ETH Transfer | 21,000 | Send ETH between accounts |
| ERC-20 Token Transfer | ~65,000 | Send tokens (e.g., USDC) |
| Uniswap Swap | ~150,000 | Trade tokens on a DEX |
| NFT Mint | ~200,000 | Create a new NFT |
| Contract Deployment | ~1,000,000+ | Deploy a new smart contract |
| SSTORE (write storage) | 20,000 | Write a value to contract storage |
| SLOAD (read storage) | 800 | Read a value from storage |
| ADD (arithmetic) | 3 | Simple math operation |

In [ ]:
# Compare transaction costs across different operation types
operations = {
    'ETH Transfer': 21000,
    'Token Transfer': 65000,
    'Uniswap Swap': 150000,
    'NFT Mint': 200000,
    'Contract Deploy': 1000000,
}

print("=" * 70)
print(f"COST COMPARISON (at {gas_price_gwei} Gwei, ETH = ${eth_price_usd:,})")
print("=" * 70)
print(f"\n{'Operation':<20} {'Gas Used':>12} {'Cost (ETH)':>14} {'Cost (USD)':>12}")
print("-" * 60)

costs_usd = []
labels = []
for op, gas in operations.items():
    cost_eth = gas * gas_price_gwei * 1e-9
    cost_usd = cost_eth * eth_price_usd
    costs_usd.append(cost_usd)
    labels.append(op)
    print(f"{op:<20} {gas:>12,} {cost_eth:>14.6f} ${cost_usd:>10.2f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2CA02C', '#0066CC', '#FF7F0E', '#D62728', '#3333B2']
bars = ax.barh(labels, costs_usd, color=colors)

for bar, cost in zip(bars, costs_usd):
    ax.text(bar.get_width() + max(costs_usd) * 0.02, bar.get_y() + bar.get_height()/2,
            f'${cost:.2f}', va='center', fontsize=10)

ax.set_xlabel('Cost (USD)', fontsize=12)
ax.set_title(f'Transaction Costs at {gas_price_gwei} Gwei (ETH = ${eth_price_usd:,})', fontsize=14)
plt.tight_layout()
plt.show()

## EIP-1559: The Modern Fee Model

Since the London upgrade (August 2021), Ethereum uses EIP-1559:

```
Total Fee = (Base Fee + Priority Tip) × Gas Used
```

- **Base Fee**: Set by the network, adjusts based on demand. This portion is **burned** (destroyed)
- **Priority Tip**: Set by the user, goes to validators. Higher tip = faster inclusion
- **Max Fee**: Maximum the user is willing to pay per gas unit

The base fee adjusts automatically:
- Block > 50% full → base fee increases (up to 12.5%)
- Block < 50% full → base fee decreases (up to 12.5%)

In [ ]:
# ========================================
# EIP-1559 PARAMETERS - CHANGE THESE!
# ========================================
base_fee_gwei = 30        # Network-determined base fee
priority_tip_gwei = 2     # Your tip to the validator
max_fee_gwei = 60         # Maximum you're willing to pay
gas_used_eip = 21000      # Gas for the transaction
# ========================================

effective_price = min(base_fee_gwei + priority_tip_gwei, max_fee_gwei)
total_fee_gwei = effective_price * gas_used_eip
total_fee_eth = total_fee_gwei * 1e-9
burned_eth = base_fee_gwei * gas_used_eip * 1e-9
validator_eth = priority_tip_gwei * gas_used_eip * 1e-9
total_fee_usd = total_fee_eth * eth_price_usd

print("=" * 60)
print("EIP-1559 FEE BREAKDOWN")
print("=" * 60)
print(f"\n  Base Fee:        {base_fee_gwei:>8} Gwei (network-set, burned)")
print(f"  Priority Tip:    {priority_tip_gwei:>8} Gwei (your tip to validator)")
print(f"  Max Fee:         {max_fee_gwei:>8} Gwei (your ceiling)")
print(f"  Effective Price: {effective_price:>8} Gwei")
print(f"\n  {'─' * 44}")
print(f"  Total Fee:       {total_fee_eth:.6f} ETH (${total_fee_usd:.2f})")
print(f"    ├─ Burned:     {burned_eth:.6f} ETH (reduces supply)")
print(f"    └─ Validator:  {validator_eth:.6f} ETH (validator reward)")

if base_fee_gwei + priority_tip_gwei > max_fee_gwei:
    print(f"\n  \u26a0 Note: Effective price capped at max_fee ({max_fee_gwei} Gwei)")
print("=" * 60)

## Exercises

1. **Cost sensitivity**: At what gas price (Gwei) does a simple ETH transfer exceed $10? Change `gas_price_gwei` and `eth_price_usd` to find out.

2. **Contract deployment**: How much would deploying a smart contract (~3,000,000 gas) cost at 10 Gwei vs 100 Gwei? Change `gas_used` to 3000000 and compare.

3. **Bull vs bear market**: Compare transaction costs when ETH is $500 vs $5,000. Only change `eth_price_usd`.

4. **EIP-1559 dynamics**: What happens when you set `priority_tip_gwei` to 0? Can your transaction still be included?

5. **Fee cap protection**: Set `base_fee_gwei` to 100 but `max_fee_gwei` to 40. What effective price do you pay?

## Key Takeaways

1. **Gas Fee = Gas Used x Gas Price** - the fundamental cost model for Ethereum
2. **Gas used** is determined by operation complexity (21,000 for a simple transfer, millions for contract deployment)
3. **Gas price** fluctuates with network demand - high demand means higher prices
4. **EIP-1559** introduced a base fee (burned) + priority tip (to validators) model
5. **The base fee auto-adjusts** to target 50% block utilization
6. **Token burning** from EIP-1559 makes ETH potentially deflationary
7. **Understanding gas** is essential for budgeting transaction costs and optimizing smart contracts